In [3]:


import os
import numpy as np
import tensorflow as tf

from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

In [4]:
#cnn feature extraction
cnn=MobileNetV2(
    weights="imagenet",
    include_top=False,
    pooling="avg"
)

cnn.trainable=False

C:\Users\darsh\AppData\Local\Temp\ipykernel_10468\2996189653.py:2: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  cnn=MobileNetV2(


In [5]:
#mock input, in future we will loop this over the entire directory to process all 100 signs
sequence_path=(r"C:\Users\darsh\Documents\ML_Datasets\wlasl\preprocessing\val\pose\woman\68764")

image_paths=sorted([
    os.path.join(sequence_path,filename)
    for filename in os.listdir(sequence_path)
    if filename.lower().endswith(".jpg")
])

In [6]:
#load frames
frames=[]

for image_path in image_paths:
    image=tf.keras.utils.load_img(
        image_path,
        target_size=(224,224)
    )

    image=tf.keras.utils.img_to_array(image)

    frames.append(image)

frames=np.array(frames)

print("Frames shape: ",frames.shape)

Frames shape:  (16, 224, 224, 3)


In [7]:
#cnn preprocessing
frames=preprocess_input(frames)

#extract features
features=cnn.predict(frames)

print("Feature sequence shapes: ",features.shape)

1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step
Feature sequence shapes:  (16, 1280)


In [8]:
#gru
gru=tf.keras.layers.GRU(128)

#inputs
inputs=tf.keras.Input(shape=(None, 512))

gru_output=gru(inputs)

print("GRU output shape: ",gru_output.shape)

#output
outputs=tf.keras.layers.Dense(
    100,
    activation="softmax"
)(gru_output)

model=tf.keras.Model(inputs=inputs, outputs=outputs)

model.summary()

GRU output shape:  (None, 128)


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)           │ (None, None, 512)           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ gru (GRU)                            │ (None, 128)                 │         246,528 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 100)                 │          12,900 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 259,428 (1013.39 KB)

 Trainable params: 259,428 (1013.39 KB)

 Non-trainable params: 0 (0.00 B)

In [9]:
#iterating through the dataset

import os

#dataset path
DATASET_ROOT=r"C:\Users\darsh\Documents\ML_Datasets\wlasl\preprocessing"

dataset=[]

for split in ["train", "val", "test"]:

    print(f"\n========== {split.upper()} ==========")

    frames_path=os.path.join(
        DATASET_ROOT,
        split,
        "frames"
    )

    for word in os.listdir(frames_path):
        word_path=os.path.join(
            frames_path,
            word
        )

        if not os.path.isdir(word_path):
            continue

        for sequence in os.listdir(word_path):
            sequence_path=os.path.join(
                word_path,
                sequence
            )

            if not os.path.isdir(sequence_path):
                continue

            #all image paths
            image_paths=[]

            for image in sorted(os.listdir(sequence_path)):
                if image.lower().endswith(".jpg"):
                    image_path=os.path.join(
                        sequence_path,
                        image
                    )

                    image_paths.append(image_path)

            dataset.append({
                "split":split,
                "word":word,
                "sequence":sequence,
                "image_paths":image_paths
            })

print("Total sequence: ",len(dataset))

print(dataset[0])



========== TRAIN ==========

========== VAL ==========

========== TEST ==========
Total sequence:  2035
{'split': 'train', 'word': 'accident', 'sequence': '00618', 'image_paths': ['C:\\Users\\darsh\\Documents\\ML_Datasets\\wlasl\\preprocessing\\train\\frames\\accident\\00618\\accident_0.jpg', 'C:\\Users\\darsh\\Documents\\ML_Datasets\\wlasl\\preprocessing\\train\\frames\\accident\\00618\\accident_1.jpg', 'C:\\Users\\darsh\\Documents\\ML_Datasets\\wlasl\\preprocessing\\train\\frames\\accident\\00618\\accident_10.jpg', 'C:\\Users\\darsh\\Documents\\ML_Datasets\\wlasl\\preprocessing\\train\\frames\\accident\\00618\\accident_11.jpg', 'C:\\Users\\darsh\\Documents\\ML_Datasets\\wlasl\\preprocessing\\train\\frames\\accident\\00618\\accident_12.jpg', 'C:\\Users\\darsh\\Documents\\ML_Datasets\\wlasl\\preprocessing\\train\\frames\\accident\\00618\\accident_13.jpg', 'C:\\Users\\darsh\\Documents\\ML_Datasets\\wlasl\\preprocessing\\train\\frames\\accident\\00618\\accident_14.jpg', 'C:\\Users\\dar

In [10]:
import tensorflow as tf
import numpy as np
def load_sequence_images(image_paths):
    frames=[]

    for image in image_paths:
        image=tf.keras.utils.load_img(
            image_path,
            target_size=(224,224)
        )

        image=tf.keras.utils.img_to_array(image)

        frames.append(image)

    return np.array(frames, dtype=np.float32)



In [12]:
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

FEATURES_ROOT=r"C:\Users\darsh\GRU_for_signframe\cnn_extracted_features"

def extract_cnn_features(item):

    #load all images in this sequence
    frames=load_sequence_images(
        item["image_paths"]
    )

    #preprocess for MobileNetV2
    frames=preprocess_input(frames)

    #extract CNN features
    features=cnn.predict(
        frames,
        verbose=0
    )

    #create directory
    save_dir=os.path.join(
        FEATURES_ROOT,
        item["split"],
        item["word"]
    )

    os.makedirs(save_dir, exist_ok=True)

    #create path
    save_path=os.path.join(
        save_dir,
        f"{item['sequence']}.npy"
    )

    #save extracted features
    np.save(save_path, features) 

    return features

In [13]:
def process_split(split_name):
    X=[]
    y=[]

    for item in dataset:

        if item["split"]!=split_name:
            continue

        features=extract_cnn_features(
            item
        )

        #store features
        X.append(features)

        #store word label
        y.append(item["word"])

    return X,y

In [ ]:
#this is just for demonstration, to test complete splits ie test, train and val

from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

X_train, y_train = process_split("train")

X_val, y_val = process_split("val")

X_test, y_test = process_split("test")

print("Training sequences:", len(X_train))
print("Validation sequences:", len(X_val))
print("Testing sequences:", len(X_test))

In [15]:
print(len(X_train) if "X_train" in globals() else "X_train not created")
print(len(X_val) if "X_val" in globals() else "X_val not created")
print(len(X_test) if "X_test" in globals() else "X_test not created")

1440
337
258


In [16]:
sample_features = np.load(
    r"C:\Users\darsh\GRU_for_signframe\cnn_extracted_features\train\africa\01384.npy"
)

print(sample_features.shape)
print(sample_features.dtype)

(16, 1280)
float32
